# 📲 Passerelle OTP WhatsApp — 100 % gratuite (Google Colab)

Ce notebook héberge **gratuitement** la passerelle qui envoie les codes OTP
de ta boutique **depuis ton numéro WhatsApp** (aucun abonnement).

**Comment ça marche :**
1. Il installe Node + la passerelle (`wa-gateway`, Baileys).
2. Il restaure ta session WhatsApp depuis **Google Drive** (un seul scan QR,
   les relances suivantes n'en ont plus besoin).
3. Il expose la passerelle sur internet via un **tunnel Cloudflare gratuit**,
   pour que ton site (LWS) puisse l'appeler.

**À la fin, le notebook affiche les 3 lignes à mettre dans le `.env` Laravel.**

> ⚠️ Contraintes du gratuit Colab : **garde cet onglet ouvert** (sinon coupure),
> session **~12 h max**. À chaque relance : *Exécution > Tout exécuter* (+ mettre
> à jour `WHATSAPP_GATEWAY_URL` dans le `.env`, l'adresse change à chaque fois).


In [ ]:
# ====== CONFIG (modifie le jeton !) ======
GATEWAY_TOKEN = "change-moi-un-jeton-long-et-secret"
PORT = 3001
PAIR_NUMBER = ""  # optionnel : ex "22382019583" -> lier avec un CODE au lieu du QR
WORKDIR = "/content/wa-gateway"
DRIVE_AUTH_DIR = "/content/drive/MyDrive/wa-gateway-auth"
CLOUDFLARE_TOKEN = ""  # optionnel : tunnel FIXE (voir fin du notebook)
CLOUDFLARE_HOSTNAME = ""  # ex "wa.maboutique.com" (avec le token ci-dessus)
# ===========================================
import os
os.environ["GATEWAY_TOKEN"] = GATEWAY_TOKEN
os.environ["PAIR_NUMBER"] = PAIR_NUMBER
os.environ["CLOUDFLARE_TOKEN"] = CLOUDFLARE_TOKEN
os.environ["CLOUDFLARE_HOSTNAME"] = CLOUDFLARE_HOSTNAME
print("Config OK.",
      "⚠️ PENSE À CHANGER LE JETON !" if "change-moi" in GATEWAY_TOKEN else "✔ jeton personnalisé")


In [ ]:
%%bash
set -e
if command -v node >/dev/null && [ "$(node -p 'process.versions.node.split(".")[0]')" -ge 18 ]; then
  node -v; npm -v; echo "Node OK"
else
  echo "Installation Node 20..."
  cd /tmp
  curl -fsSLO https://nodejs.org/dist/v20.18.0/node-v20.18.0-linux-x64.tar.xz
  tar -xJf node-v20.18.0-linux-x64.tar.xz -C /usr/local --strip-components=1
  node -v; npm -v
fi

In [ ]:
"""Écrit les fichiers de la passerelle (même code que wa-gateway/ du projet)."""
import os
WORKDIR_FALLBACK = '/content/wa-gateway'
WORKDIR = globals().get('WORKDIR', WORKDIR_FALLBACK)
os.makedirs(WORKDIR, exist_ok=True)
PACKAGE_JSON = '{\n    "name": "boutique-wa-gateway",\n    "version": "1.0.0",\n    "private": true,\n    "description": "Passerelle WhatsApp auto-hébergée (100% gratuite) : envoie les codes OTP du site depuis ton propre numéro.",\n    "type": "module",\n    "scripts": {\n        "start": "node server.js"\n    },\n    "dependencies": {\n        "@whiskeysockets/baileys": "^6.7.18",\n        "express": "^4.21.2",\n        "qrcode": "^1.5.4"\n    }\n}\n'
SERVER_JS = '// Passerelle WhatsApp 100% gratuite pour la Boutique.\n// Le site génère le code OTP, cette passerelle l\'envoie depuis TON numéro.\n//\n// 1. npm install\n// 2. GATEWAY_TOKEN=change-moi node server.js   (ou : voir .env.example)\n// 3. Scanne le QR affiché (ou ouvre http://localhost:3001/qr) avec WhatsApp :\n//    Paramètres > Appareils liés > Lier un appareil.\n// 4. Côté Laravel (.env) : WHATSAPP_OTP_DRIVER=gateway\n//    + WHATSAPP_GATEWAY_URL=http://127.0.0.1:3001/send (dev)\n//    (+ WHATSAPP_GATEWAY_TOKEN=change-moi)\n//\n// La session est conservée dans ./auth : un seul scan suffit (sauf déconnexion).\nimport express from \'express\';\nimport makeWASocket, {\n    DisconnectReason,\n    fetchLatestBaileysVersion,\n    useMultiFileAuthState,\n} from \'@whiskeysockets/baileys\';\nimport QRCode from \'qrcode\';\nimport fs from \'node:fs\';\n\nconst PORT = Number(process.env.PORT || 3001);\nconst TOKEN = process.env.GATEWAY_TOKEN || \'\';\nconst PAIR_NUMBER = (process.env.PAIR_NUMBER || \'\').replace(/\\D+/g, \'\');\n\nconst app = express();\napp.use(express.json({ limit: \'256kb\' }));\n\nlet sock = null;\nlet connected = false;\nlet lastQr = null;\nlet lastQrAt = 0;\n\nfunction checkToken(req, res, next) {\n    if (!TOKEN) return next(); // dev local sans token\n    const given =\n        req.body?.token ?? req.query?.token ?? req.headers[\'x-gateway-token\'] ?? \'\';\n    if (given !== TOKEN) {\n        return res.status(401).json({ sent: false, error: \'bad token\' });\n    }\n    next();\n}\n\nasync function start() {\n    const { state, saveCreds } = await useMultiFileAuthState(\'./auth\');\n    let version;\n    try {\n        ({ version } = await fetchLatestBaileysVersion());\n    } catch {\n        version = undefined;\n    }\n\n    sock = makeWASocket({\n        auth: state,\n        version,\n        printQRInTerminal: false,\n        browser: [\'Boutique OTP\', \'Chrome\', \'1.0\'],\n    });\n    sock.ev.on(\'creds.update\', saveCreds);\n\n    sock.ev.on(\'connection.update\', (u) => {\n        const { connection, lastDisconnect, qr } = u;\n        if (qr) {\n            lastQr = qr;\n            lastQrAt = Date.now();\n            console.log(\'--- Scanne ce QR avec WhatsApp (Appareils liés > Lier un appareil) ---\');\n            console.log(qr);\n            console.log(`--- ou ouvre http://localhost:${PORT}/qr ---`);\n        }\n        if (connection === \'open\') {\n            connected = true;\n            lastQr = null;\n            console.log(\'WhatsApp connecté, prêt à envoyer les codes OTP.\');\n        }\n        if (connection === \'close\') {\n            connected = false;\n            const code = lastDisconnect?.error?.output?.statusCode;\n            const loggedOut = code === DisconnectReason.loggedOut;\n            console.log(\n                loggedOut\n                    ? \'Session supprimée côté WhatsApp : supprime ./auth et rescanner le QR.\'\n                    : \'Connexion perdue, reconnexion...\',\n            );\n            if (loggedOut) {\n                try {\n                    fs.rmSync(\'./auth\', { recursive: true, force: true });\n                } catch {}\n            }\n            setTimeout(start, loggedOut ? 3000 : 5000);\n        }\n    });\n\n    // Jumelage par code (pratique sur serveur distant sans écran) : le code\n    // s\'affiche ici, à saisir dans WhatsApp > Appareils liés > Lier avec un code.\n    if (PAIR_NUMBER && !state.creds.registered) {\n        setTimeout(async () => {\n            try {\n                const pairingCode = await sock.requestPairingCode(PAIR_NUMBER);\n                console.log(`Code de jumelage pour ${PAIR_NUMBER} : ${pairingCode}`);\n            } catch (e) {\n                console.log(\'Jumelage par code impossible :\', e?.message || e);\n            }\n        }, 8000);\n    }\n}\n\n// Accepte « 22379000000 », « +223... », « 223...@s.whatsapp.net ».\nfunction toJid(to) {\n    let d = String(to || \'\').replace(/\\D+/g, \'\');\n    if (d.startsWith(\'00223\')) d = d.slice(2);\n    return d + \'@s.whatsapp.net\';\n}\n\napp.get(\'/status\', (req, res) => {\n    res.json({ connected, qrAvailable: !!lastQr });\n});\n\napp.get(\'/qr\', async (req, res) => {\n    // Le QR permet de lier TON numéro : protégé par token si configuré\n    // (utile quand la passerelle est exposée via tunnel, ex Colab).\n    if (TOKEN) {\n        const given = req.query?.token ?? req.headers[\'x-gateway-token\'] ?? \'\';\n        if (given !== TOKEN) {\n            return res.status(401).json({ error: \'token requis (?token=...)\' });\n        }\n    }\n    if (!lastQr) {\n        return res\n            .status(connected ? 410 : 404)\n            .json({ error: connected ? \'déjà connecté\' : \'QR pas encore généré, réessaie dans 10 s\' });\n    }\n    try {\n        const img = await QRCode.toDataURL(lastQr, { width: 280, margin: 1 });\n        res.send(\n            `<!doctype html><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">` +\n                `<body style="font-family:sans-serif;text-align:center;padding:24px">` +\n                `<h2>Lier ton numéro WhatsApp</h2>` +\n                `<p>WhatsApp > Paramètres > Appareils liés > Lier un appareil</p>` +\n                `<img src="${img}" alt="QR WhatsApp"><script>setTimeout(()=>location.reload(),20000)</script>`,\n        );\n    } catch {\n        res.status(500).json({ error: \'QR illisible\' });\n    }\n});\n\napp.post(\'/send\', checkToken, async (req, res) => {\n    const { to, body } = req.body || {};\n    if (!to || !body) {\n        return res.status(422).json({ sent: false, error: \'to/body requis\' });\n    }\n    if (!sock || !connected) {\n        return res.status(503).json({ sent: false, error: \'WhatsApp non connecté (scanne le QR)\' });\n    }\n    try {\n        const r = await sock.sendMessage(toJid(to), { text: String(body) });\n        res.json({ sent: true, id: r?.key?.id || null });\n    } catch (e) {\n        res.status(502).json({ sent: false, error: String(e?.message || e) });\n    }\n});\n\napp.listen(PORT, () => {\n    console.log(`Passerelle OTP : http://localhost:${PORT}  (/status, /qr, POST /send)`);\n    start();\n});\n'
open(os.path.join(WORKDIR, "package.json"), "w").write(PACKAGE_JSON)
open(os.path.join(WORKDIR, "server.js"), "w").write(SERVER_JS)
print("package.json + server.js écrits dans", WORKDIR)


In [ ]:
from google.colab import drive
import os, shutil, subprocess

drive.mount('/content/drive', force_remount=False)

auth = os.path.join(WORKDIR, 'auth')
if os.path.isdir(DRIVE_AUTH_DIR) and os.listdir(DRIVE_AUTH_DIR):
    shutil.rmtree(auth, ignore_errors=True)
    shutil.copytree(DRIVE_AUTH_DIR, auth)
    print("Session WhatsApp restaurée depuis Drive (pas de rescan normalement).")
else:
    print("Première fois : tu scannera le QR à l'étape suivante (une seule fois).")

r = subprocess.run("npm install --no-audit --no-fund", shell=True, cwd=WORKDIR,
                   capture_output=True, text=True)
print("npm install :", "OK" if r.returncode == 0 else "ÉCHEC")
print((r.stdout + r.stderr)[-800:])

In [ ]:
import os, re, subprocess, time

subprocess.run(r"pkill -f 'node server\.js' || true", shell=True)
subprocess.run(r"pkill -f 'cloudflared tunnel' || true", shell=True)
time.sleep(2)

gw_log = os.path.join(WORKDIR, "gateway.log")
env = dict(os.environ, PORT=str(PORT), GATEWAY_TOKEN=GATEWAY_TOKEN)
gw = subprocess.Popen(["node", "server.js"], cwd=WORKDIR,
                      stdout=open(gw_log, "w"), stderr=subprocess.STDOUT, env=env)
print("Passerelle démarrée, PID", gw.pid, "(log: gateway.log)")

cf = os.path.join(WORKDIR, "cloudflared")
if not os.path.exists(cf):
    print("Téléchargement cloudflared...")
    subprocess.run(
        f"curl -fsSL -o {cf} "
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        shell=True, check=True)
    os.chmod(cf, 0o755)

cf_log = os.path.join(WORKDIR, "cloudflared.log")
open(cf_log, "w").close()
subprocess.Popen([cf, "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
                 stdout=open(cf_log, "w"), stderr=subprocess.STDOUT)

CF_TOKEN = os.environ.get("CLOUDFLARE_TOKEN", "")
CF_HOST = os.environ.get("CLOUDFLARE_HOSTNAME", "")
url = None
if CF_TOKEN:
    subprocess.Popen([cf, "tunnel", "--no-autoupdate", "run", "--token", CF_TOKEN],
                     stdout=open(cf_log, "w"), stderr=subprocess.STDOUT)
    time.sleep(8)
    if not CF_HOST:
        raise SystemExit("CLOUDFLARE_HOSTNAME vide : renseigne ton adresse fixe (ex wa.maboutique.com).")
    url = "https://" + CF_HOST
    print("Tunnel FIXE :", url, "(stable entre les relances, pas de .env à refaire)")
else:
    for _ in range(90):
        time.sleep(1)
        with open(cf_log, errors="ignore") as f:
            m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", f.read())
        if m:
            url = m.group(0)
            break
    if not url:
        raise SystemExit("Tunnel non établi après 90 s. Regarde cloudflared.log dans " + WORKDIR)

open(os.path.join(WORKDIR, "tunnel_url.txt"), "w").write(url)
print("URL publique :", url)
print("Vérif :", url + "/status")

In [ ]:
import os, re, shutil, subprocess, sys, time
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "qrcode", "pillow", "requests"],
               check=True)
import requests
from IPython.display import HTML, display

base = f"http://127.0.0.1:{PORT}"
shown = None
pair_shown = False
gw_log = os.path.join(WORKDIR, "gateway.log")
deadline = time.time() + 300
connected = False
while time.time() < deadline:
    try:
        s = requests.get(base + "/status", timeout=5).json()
        if s.get("connected"):
            connected = True
            break
        if PAIR_NUMBER and not pair_shown:
            try:
                log = open(gw_log, errors="ignore").read()
                m2 = re.search(r"Code de jumelage pour \S+ : (\S+)", log)
                if m2:
                    pair_shown = True
                    display(HTML(f"<h3>Code de jumelage : <b style='font-size:28px'>{m2.group(1)}</b></h3><p>WhatsApp &gt; Appareils liés &gt; Lier avec un code</p>"))
            except Exception:
                pass
        r = requests.get(base + "/qr", params={"token": GATEWAY_TOKEN}, timeout=5)
        if r.status_code == 200 and "data:image" in r.text:
            m = re.search(r'src="(data:image/[^"]+)"', r.text)
            if m and m.group(1) != shown:
                shown = m.group(1)
                display(HTML("<h3>Scanne ce QR (ou utilise le code de jumelage, voir gateway.log) : WhatsApp &gt; Appareils liés &gt; Lier un appareil</h3>"
                             f"<img src='{shown}' width='280'>"))
    except Exception:
        pass
    time.sleep(5)

if not connected:
    raise SystemExit("Pas connecté après 5 min. Relance cette cellule (ou regarde gateway.log).")

print("✅ WhatsApp connecté ! Sauvegarde de la session sur Drive...")
shutil.rmtree(DRIVE_AUTH_DIR, ignore_errors=True)
shutil.copytree(os.path.join(WORKDIR, "auth"), DRIVE_AUTH_DIR)

url = open(os.path.join(WORKDIR, "tunnel_url.txt")).read().strip()
print()
print("Mets ceci dans le .env Laravel (puis : php artisan config:cache) :")
print("WHATSAPP_OTP_DRIVER=gateway")
print(f"WHATSAPP_GATEWAY_URL={url}/send")
print(f"WHATSAPP_GATEWAY_TOKEN={GATEWAY_TOKEN}")
print()
print("Test d'envoi manuel : remplace le numéro et ouvre dans ton navigateur :")
print(f"(non — POST uniquement ; le site s'en charge tout seul)")

## ✅ C'est en ligne !

Tant que cet onglet reste ouvert, chaque code OTP du site part **depuis ton
numéro, gratuitement**.

**À retenir :**
- **Relance** (*Exécution > Tout exécuter*) si Colab coupe (~12 h max, ou
  onglet fermé) : pas de rescan QR grâce à Drive, mais **l'adresse du tunnel
  change** → mets à jour `WHATSAPP_GATEWAY_URL` dans le `.env` + `config:cache`.
- **Ne partage jamais** `GATEWAY_TOKEN` ni l'adresse du tunnel : quiconque les
  a peut envoyer des WhatsApp avec ton numéro.
- Si WhatsApp dissocie l'appareil un jour : supprime le dossier
  `wa-gateway-auth` sur ton Drive et rescane le QR.
- Dépannage : `gateway.log` et `cloudflared.log` sont dans `/content/wa-gateway`.
